# SegOCR — Colab Quickstart

Run SegOCR end to end on a free Colab GPU runtime.

**Setup**: `Runtime → Change runtime type → T4 GPU` (free) or any A100/L4 (paid).

This notebook covers:
1. Clone the repo + install deps.
2. Generate a small synthetic dataset using bundled corpora and Linux system fonts.
3. Visually inspect generated samples.
4. Train a UNet+ResNet18 baseline for ~5 minutes.
5. Run inference and overlay predictions.

Total runtime end-to-end: ~10–15 minutes on T4.

## 1 · Clone + install

Replace the `git clone` URL with your fork. Colab already ships with PyTorch + CUDA, so we only install the SegOCR-specific deps.

In [ ]:
# Clone the repo (replace with your fork's URL)
!git clone https://github.com/mauryantitans/SegOCR.git /content/segocr
%cd /content/segocr

In [ ]:
# Install the SegOCR package + base deps. Torch is already on Colab.
!pip install -q -e .
!pip install -q -r requirements/base.txt
!pip install -q segmentation-models-pytorch wandb

In [ ]:
# Sanity-check: imports + tests should pass.
!python -m pytest tests/ -q --tb=no 2>&1 | tail -5

## 2 · Verify GPU

Should print `cuda` and the GPU name. If it prints `cpu`, switch the runtime to a GPU before continuing — training on CPU will work but be ~50× slower.

In [ ]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 3 · Generate a small synthetic dataset

We use Linux system fonts (`/usr/share/fonts`) since Colab doesn't have Google Fonts cloned by default. The bundled mini-corpora (`signs`, `receipts`, `names`, `numbers`) ship with the package, so generated text is semantically meaningful out of the box.

For a real run, also grab COCO via `bash scripts/setup_data.sh` (~25 GB) for Tier-3 backgrounds. We skip that here for speed.

In [ ]:
import yaml
from pathlib import Path

# Build a Colab-friendly config from the default one
cfg = yaml.safe_load(Path('segocr/configs/default.yaml').read_text())
cfg['generator']['fonts']['root_dir'] = '/usr/share/fonts'
cfg['generator']['fonts']['cache_path'] = '/tmp/segocr_font_cache.json'
cfg['generator']['image_size'] = [256, 256]
cfg['generator']['fonts']['min_size'] = 20
cfg['generator']['fonts']['max_size'] = 64
cfg['generator']['background']['natural_image_dirs'] = []
cfg['generator']['num_workers'] = 2
# Drop the wikitext corpus reference (not downloaded); keep the bundled ones.
cfg['generator']['text']['corpus_paths'] = [
    {'path': 'BUNDLED:signs', 'tag': 'signs', 'weight': 0.30},
    {'path': 'BUNDLED:receipts', 'tag': 'receipts', 'weight': 0.20},
    {'path': 'BUNDLED:names', 'tag': 'names', 'weight': 0.20},
    {'path': 'BUNDLED:numbers', 'tag': 'numbers', 'weight': 0.30},
]
Path('/content/colab_config.yaml').write_text(yaml.safe_dump(cfg))
print('Config written to /content/colab_config.yaml')

In [ ]:
# Generate 500 samples. ~30 seconds on Colab.
!python -m scripts.generate_dataset \
    --config /content/colab_config.yaml \
    --num-images 500 \
    --output /content/segocr_data

## 4 · Inspect generated samples

Eyeball a few images side by side with their semantic mask overlays. Pixels inside the masked region are tinted red.

In [ ]:
import json
import cv2
import matplotlib.pyplot as plt

n_show = 8
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i, ax in enumerate(axes.flat):
    name = f'{i:06d}'
    img = cv2.cvtColor(
        cv2.imread(f'/content/segocr_data/images/{name}.png'),
        cv2.COLOR_BGR2RGB,
    )
    sem = cv2.imread(f'/content/segocr_data/semantic/{name}.png', cv2.IMREAD_GRAYSCALE)
    overlay = img.copy()
    overlay[sem > 0] = [255, 0, 0]
    blended = cv2.addWeighted(img, 0.6, overlay, 0.4, 0)
    meta = json.loads(open(f'/content/segocr_data/metadata/{name}.json').read())
    text = ''.join(c['char'] for c in meta['characters'])[:30]
    ax.imshow(blended)
    ax.set_title(f"{meta['layout_mode']}: {text}", fontsize=10)
    ax.axis('off')
plt.tight_layout()
plt.show()

## 5 · Train a UNet baseline

Trains a UNet with ResNet-18 encoder for 500 iterations. ~3–5 minutes on T4. The training loop reads from the dataset we just generated and reports per-class IoU + foreground/binary mIoU at every eval step.

In [ ]:
# Trim the config for fast Colab training
cfg['generator']['output_dir'] = '/content/segocr_data'
cfg['model']['architecture'] = 'unet'
cfg['model']['encoder'] = 'resnet18'    # smaller than resnet50 for speed
cfg['model']['head_features'] = 32
cfg['model']['decoder_channels'] = [128, 64, 32, 16, 16]
cfg['model']['heads'] = {'semantic': True, 'affinity': True, 'direction': True}
cfg['training']['learning_rate'] = 1e-3
cfg['training']['total_iters'] = 500
cfg['training']['warmup_iters'] = 100
cfg['training']['eval_interval'] = 100
cfg['training']['save_interval'] = 250
cfg['training']['log_interval'] = 25
cfg['training']['batch_size'] = 8
cfg['training']['num_workers'] = 2
cfg['training']['mixed_precision'] = True
cfg['training']['output_dir'] = '/content/segocr_weights'
cfg['training']['ema'] = {'enabled': True, 'decay': 0.99}
cfg['training']['wandb'] = {'project': None}  # disable wandb for the quickstart
Path('/content/colab_config.yaml').write_text(yaml.safe_dump(cfg))

In [ ]:
!python -m scripts.train_model --config /content/colab_config.yaml

## 6 · Visualize predictions

Load the best checkpoint and overlay the predicted mask on a held-out validation sample.

In [ ]:
import torch
import numpy as np
from segocr.models.unet import build_model
from segocr.training.dataset import SegOCRDataset, collate_fn
from segocr.utils.config import load_config

cfg = load_config('/content/colab_config.yaml')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = build_model(cfg['model']).to(device).eval()

# Find the latest checkpoint
ckpts = sorted(Path('/content/segocr_weights').glob('checkpoint_*.pth'))
if ckpts:
    state = torch.load(ckpts[-1], map_location=device)
    model.load_state_dict(state['model'])
    print(f'Loaded {ckpts[-1].name}')

val_ds = SegOCRDataset('/content/segocr_data', split='val', train_aug=False)
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
with torch.no_grad():
    for i, ax in enumerate(axes.flat):
        sample = val_ds[i]
        x = sample['image'].unsqueeze(0).to(device)
        out = model(x)
        pred = out['semantic'].argmax(dim=1).cpu().numpy()[0]
        # Restore image for display
        img = sample['image'].permute(1, 2, 0).cpu().numpy()
        img = (img * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406]))
        img = (np.clip(img, 0, 1) * 255).astype(np.uint8)
        overlay = img.copy()
        overlay[pred > 0] = [0, 255, 0]
        blended = cv2.addWeighted(img, 0.6, overlay, 0.4, 0)
        ax.imshow(blended)
        ax.set_title(f'pred chars: {(pred > 0).sum()}', fontsize=10)
        ax.axis('off')
plt.tight_layout()
plt.show()

## What's next

This quickstart trained on a tiny synthetic dataset; results will be limited. To run a real experiment:

- Run `bash scripts/setup_data.sh` to download Google Fonts + COCO + DTD (≈25 GB).
- Generate 100K+ samples (`--num-images 100000`).
- Train `total_iters: 80000+` with `batch_size: 16` on a stronger GPU (A100).
- See `docs/USAGE.md` for the full end-to-end recipe and CLI reference.

For the latest model (SegFormer MiT-B2), install `mmsegmentation` separately — see `requirements/train.txt`.